In [2]:
# ── Imports ──────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False


In [4]:
import os
import sys

# 1. Locate the project root (parent of the 'notebooks' directory)
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
sys.path.append(os.path.abspath("../"))

# 2. Add the 'src' directory to the system path so Python can find your module
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

# 3. Import your function (matches the filename src/data_loader.py)
from src.data_loader import fetch_and_save_data

# 4. Define the correct absolute path to data/raw from the project root
save_directory = os.path.join(project_root, 'data', 'raw')

# 5. Define parameters
ALL_TICKERS = ['TSLA', 'BND', 'SPY']
START_DATE  = '2015-01-01'
END_DATE    = '2026-06-30'

# 6. Call the function with the correctly mapped save_dir
portfolio_dataframes = fetch_and_save_data(
    tickers=ALL_TICKERS, 
    start=START_DATE, 
    end=END_DATE,
    save_dir=save_directory
)

Ticker        : TSLA
Portfolio     : ['TSLA', 'BND', 'SPY']
Full period   : 2015-01-01 → 2026-06-30
Training ends : 2024-12-31
----------------------------------------
Fetching data for TSLA...
  ↳ Successfully saved to c:\Users\Hermela\Desktop\10academy\week 9\data\raw\TSLA_historical.csv
Fetching data for BND...
  ↳ Successfully saved to c:\Users\Hermela\Desktop\10academy\week 9\data\raw\BND_historical.csv
Fetching data for SPY...
  ↳ Successfully saved to c:\Users\Hermela\Desktop\10academy\week 9\data\raw\SPY_historical.csv


In [6]:
import pandas as pd
import os

# 1. Define paths relative to the notebooks folder
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
raw_dir = os.path.join(project_root, 'data', 'raw')
processed_dir = os.path.join(project_root, 'data', 'processed')

# Ensure the processed directory exists
os.makedirs(processed_dir, exist_ok=True)

tickers = ['TSLA', 'BND', 'SPY']
dataframes = {}

print("Setup complete. Ready to load data.")

Setup complete. Ready to load data.


In [15]:
for ticker in tickers:
    file_path = os.path.join(raw_dir, f"{ticker}_historical.csv")
    try:
        df = pd.read_csv(file_path)
        
        # ── FIX FOR YFINANCE MULTI-INDEX UPDATE ──
        # If 'Date' is missing, it means the first column is the date but named incorrectly
        if 'Date' not in df.columns:
            # 1. Rename the very first column to 'Date'
            df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
            
            # 2. Drop the extra text rows yfinance adds (like 'Ticker' or 'Date' labels)
            df = df[~df['Date'].isin(['Ticker', 'Date'])]
            
        dataframes[ticker] = df
        print(f"Successfully loaded {ticker} data. Shape: {dataframes[ticker].shape}")
        
    except FileNotFoundError:
        print(f"Error: Could not find {file_path}. Did you run the data loader?")

Successfully loaded TSLA data. Shape: (2888, 6)
Successfully loaded BND data. Shape: (2888, 6)
Successfully loaded SPY data. Shape: (2888, 6)


In [16]:
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

for ticker, df in dataframes.items():
    print(f"\n[{ticker} - Data Types Before]")
    print(df.dtypes)
    
    # Convert 'Date' to datetime and set as index
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
  
    # Force price and volume columns to numeric (coerces errors to NaN)
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    print(f"\n[{ticker} - Data Types After]")
    print(df.dtypes)


[TSLA - Data Types Before]
Date      str
Close     str
High      str
Low       str
Open      str
Volume    str
dtype: object

[TSLA - Data Types After]
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

[BND - Data Types Before]
Date      str
Close     str
High      str
Low       str
Open      str
Volume    str
dtype: object

[BND - Data Types After]
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

[SPY - Data Types Before]
Date      str
Close     str
High      str
Low       str
Open      str
Volume    str
dtype: object

[SPY - Data Types After]
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object


In [17]:
for ticker, df in dataframes.items():
    missing_initial = df.isnull().sum().sum()
    print(f"\n[{ticker}] Missing values before cleaning: {missing_initial}")
    
    if missing_initial > 0:
        # Interpolate based on time, then fill any edge cases
        df.interpolate(method='time', inplace=True)
        df.bfill(inplace=True)
        df.ffill(inplace=True)
        
    missing_final = df.isnull().sum().sum()
    print(f"[{ticker}] Missing values after cleaning:  {missing_final}")


[TSLA] Missing values before cleaning: 0
[TSLA] Missing values after cleaning:  0

[BND] Missing values before cleaning: 0
[BND] Missing values after cleaning:  0

[SPY] Missing values before cleaning: 0
[SPY] Missing values after cleaning:  0


In [18]:
for ticker, df in dataframes.items():
    print(f"\n{'='*40}")
    print(f" {ticker} - Basic Statistics")
    print(f"{'='*40}")
    # .round(2) formats the numbers to 2 decimal places for easier reading
    display(df.describe().round(2))


 TSLA - Basic Statistics


,Close,High,Low,Open,Volume
count,2888.00,2888.00,2888.00,2888.00,2.888000e+03
mean,148.77,151.99,145.42,148.80,1.087922e+08
std,138.90,141.85,135.87,138.98,7.082549e+07
min,9.58,10.33,9.40,9.49,1.062000e+07
25%,18.39,18.67,18.02,18.39,6.548325e+07
50%,133.44,136.05,125.83,131.50,9.033615e+07
75%,251.93,257.49,245.83,251.68,1.261204e+08
max,489.88,498.83,485.33,489.88,9.140820e+08



 BND - Basic Statistics


,Close,High,Low,Open,Volume
count,2888.00,2888.00,2888.00,2888.00,2888.00
mean,66.50,66.60,66.40,66.51,4653785.80
std,4.71,4.72,4.71,4.71,3017703.95
min,58.73,58.80,58.69,58.76,0.00
25%,62.48,62.54,62.41,62.47,2233700.00
50%,65.73,65.85,65.60,65.71,4280650.00
75%,70.69,70.84,70.55,70.69,6246475.00
max,74.83,74.92,74.80,74.89,33963000.00



 SPY - Basic Statistics


,Close,High,Low,Open,Volume
count,2888.00,2888.00,2888.00,2888.00,2.888000e+03
mean,351.51,353.34,349.37,351.44,8.551049e+07
std,155.44,156.15,154.58,155.41,4.338553e+07
min,154.16,155.21,152.07,153.72,2.027000e+07
25%,223.55,224.83,222.17,223.47,5.836455e+07
50%,312.82,315.49,310.49,314.14,7.541950e+07
75%,432.81,434.99,430.31,432.60,9.882245e+07
max,757.62,758.45,754.81,756.20,5.072443e+08


In [19]:
for ticker, df in dataframes.items():
    save_path = os.path.join(processed_dir, f"{ticker}_cleaned.csv")
    df.to_csv(save_path)
    print(f"Saved cleaned {ticker} data to: {save_path}")

print("\nAll data cleaned and saved successfully!")

Saved cleaned TSLA data to: c:\Users\Hermela\Desktop\10academy\week 9\data\processed\TSLA_cleaned.csv
Saved cleaned BND data to: c:\Users\Hermela\Desktop\10academy\week 9\data\processed\BND_cleaned.csv
Saved cleaned SPY data to: c:\Users\Hermela\Desktop\10academy\week 9\data\processed\SPY_cleaned.csv

All data cleaned and saved successfully!
